P1 - 6 - Cavidade 1D

Startup1D

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import lsrk4 as rk
import gll1D as gll 
import fluxo_numerico as flux
import operadores as op 

N = 4
K = 10
Nfaces = 2 
Nfp = 1

Np = N + 1

r, VX = gll.gerar_malha_gll_1d(0,1,K,N)
nos_xi, _ = gll.pontos_gll(N)

V =  op.V1D(N,nos_xi)
invV = np.linalg.inv(V)

Dr = op.matriz_diferenciacao(N,nos_xi,V)

LIFT = flux.Lift1D(Np,V)

v = flux.EtoV(K)
va = v[:,0]
vb = v[:,1]
x = np.ones((Np,1))*VX[va] + 0.5*(r+1)*(VX[vb] - VX[va])

rx, J = flux.fator_geometrico1D(x,Dr)

NODETOL = 1e-10
fmask1 = np.where(np.abs(nos_xi + 1) < NODETOL)[0]
fmask2 = np.where(np.abs(nos_xi - 1) < NODETOL)[0]
Fmask = np.vstack((fmask1, fmask2)).T
Fx = x[Fmask.flatten(), :]

nx = flux.normal_1D(K)
Fscale = 1/(J[Fmask,:])

etoe, etof = flux.Connect1D(flux.EtoV(K))

vmapM, vmapP, vmapB, mapB = flux.BuildMaps1D(Np,K,x,etoe,etof)


AdvecRHS1D

In [ ]:
def advecrhs1D(u,time,a,K):
    alpha = 1
    du = np.zeros((Nfp*Nfaces,K))
    